# Lesson18: ファインチューニング

## 事前準備

OpenAI APIキーの設定

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

ライブラリのインストール

In [ ]:
!pip install openai==1.91.0 httpx==0.28.1

## 教材サンプルコード

### Chapter3: ファインチューニングの方法

---

#### 2. JSONL形式への変換

In [ ]:
import pandas as pd
import json

csv_file_name = "学習用データセット.csv"
jsonl_file_name = "学習用データセット.jsonl"

# CSVファイルの読み込み
df = pd.read_csv(csv_file_name)

# CSVファイルの各行のデータを、Chat Completions APIに対応した辞書形式に変換する
jsonl_data = []
system_message = {
    "role": "system",
    "content": "あなたは織田信長の口調・性格で、質問に対して名言を織り交ぜて回答するアシスタントAIです。"
}
for _, row in df.iterrows():
    record = {
        "messages": [
            system_message,
            {"role": "user", "content": row[0]},
            {"role": "assistant", "content": row[1]}
        ]
    }
    jsonl_data.append(record)

# JSONL形式のファイルにCSVファイルの各行のデータを書き込む
# ※最終行のみ、改行を付けない
with open(jsonl_file_name, 'w', encoding='utf-8') as jsonl_file:
    data_len = len(jsonl_data)
    for i, entry in enumerate(jsonl_data):
        if data_len - 1 == i:
            jsonl_file.write(json.dumps(entry, ensure_ascii=False))
        else:
            jsonl_file.write(json.dumps(entry, ensure_ascii=False) + '\n')

#### 3. 事前学習済みモデルの用意

In [ ]:
from openai import OpenAI
client = OpenAI()

#### 4. 学習データのモデルへのアップロード

In [ ]:
file_obj = client.files.create(
  file=open(jsonl_file_name, "rb"),
  purpose="fine-tune"
)

#### 5. ファインチューニングの実行

In [ ]:
job = client.fine_tuning.jobs.create(
  training_file=file_obj.id,
  model="gpt-4o-mini-2024-07-18"
)

#### 6. 学習ステータスの確認

In [ ]:
fine_tuned_job = client.fine_tuning.jobs.retrieve(job.id)
fine_tuned_job

#### 7. ファインチューニング済みモデルの利用

In [ ]:
completion = client.chat.completions.create(
    model=fine_tuned_job.fine_tuned_model,
    messages=[
        {"role": "system", "content": "あなたは織田信長の口調・性格で、質問に対して名言を織り交ぜて回答するアシスタントAIです。"},
        {"role": "user", "content": "成功の秘訣は？"}
    ]
)

completion.choices[0].message.content